# Pipeline and Models

Run the full Phase 1 + 2 pipeline from **VS Code (Jupyter extension)**, terminal, or Colab.

**Local (VS Code):** Run all cells top-to-bottom. Uses `data/influencers.txt` and synthetic posts unless you set `EXTRACTED_METADATA_DIR` or `POSTS_PARQUET` below.

**Colab:** Mount Drive first, then set `EXTRACTED_METADATA_DIR` or `POSTS_PARQUET` to your 10k extract.

Legacy notebooks (`Data_Extraction.ipynb`, `Cleaned Up Notebook.ipynb`) are Colab-only exploration notebooks and were not updated.

In [ ]:
from pathlib import Path
import sys


def find_repo_root() -> Path:
    """Works from VS Code, terminal jupyter, or Colab regardless of cwd."""
    candidates: list[Path] = [Path.cwd(), *Path.cwd().parents]
    try:
        candidates.insert(0, Path(__file__).resolve().parent.parent)
    except NameError:
        pass

    colab_repo = Path("/content/drive/MyDrive/DSCI351/Class Project/DSCI_Recommender_System_Class_Proj")
    if colab_repo.exists():
        candidates.insert(0, colab_repo)

    for candidate in candidates:
        if (candidate / "src" / "pipeline.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find repo root. Open this notebook from the project repo or set cwd to notebooks/."
    )


REPO = find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DATA_DIR = REPO / "data"
OUTPUT_DIR = REPO / "artifacts"

# --- Optional overrides (leave None for local synthetic e2e run) ---
EXTRACTED_METADATA_DIR = None  # e.g. REPO / "Post_metadata_10000_extracted"
POSTS_PARQUET = None  # e.g. OUTPUT_DIR / "processed" / "posts_base_10000.parquet"

# Auto-use existing processed parquet if you already ran the pipeline once
if POSTS_PARQUET is None:
    processed_dir = OUTPUT_DIR / "processed"
    if processed_dir.exists():
        existing = sorted(processed_dir.glob("posts_base_*.parquet"))
        if existing:
            POSTS_PARQUET = existing[-1]

USE_SYNTHETIC = EXTRACTED_METADATA_DIR is None and POSTS_PARQUET is None

print("Repo:", REPO)
print("Data dir exists:", DATA_DIR.exists())
print("Influencers file exists:", (DATA_DIR / "influencers.txt").exists())
print("Mode:", "synthetic" if USE_SYNTHETIC else ("parquet" if POSTS_PARQUET else "extracted metadata"))

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(REPO / "requirements.txt"),
])
print("Dependencies installed from", REPO / "requirements.txt")

In [ ]:
from src.pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    posts_parquet=POSTS_PARQUET,
    extracted_metadata_dir=EXTRACTED_METADATA_DIR,
    synthetic=USE_SYNTHETIC,
    synthetic_influencers=120,
    k=5,
    seed=42,
)

outputs = run_pipeline(config)
outputs["results_df"]

In [ ]:
from IPython.display import Image, display

print('Hybrid alpha:', outputs['hybrid_alpha'])
print('Results saved to:', outputs['results_path'])
for figure_path in outputs['figure_paths']:
    display(Image(filename=str(figure_path)))